# Two-Stage Specialist Decoders

This notebook trains the current best two-stage model, plus annulus and two-circle specialist decoders, and a learned router from Stage 1 coefficient predictions. Outputs are written to `outputs/two_stage_specialist_decoders/`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from config import Stage1ModelConfig, Stage2ModelConfig, StageTrainingConfig, TwoStageRunConfig, TwoStageStackConfig
from datasets import build_two_stage_datasets, save_two_stage_dataset
from models import Stage1Regressor, Stage2CoordConvDecoder, evaluate_regression_predictions, evaluate_stage2_predictions, evaluate_stage2_predictions_by_shape, fit_stage1_model, fit_stage2_model, predict_stage1_coefficients, predict_stage2_logits, select_best_stage2_threshold, set_torch_seed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
set_torch_seed(SEED)

OUTPUT_ROOT = ROOT / 'outputs' / 'two_stage_specialist_decoders'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_config = TwoStageRunConfig(
    N=8,
    training_samples=10000,
    validation_samples=2000,
    test_samples=500,
    rho=0.8,
    grid_size=32,
    threshold=0.5,
    use_validation_threshold_sweep=True,
    noise_level=0.01,
    seed=SEED,
    training_shape_weights=(('two_circles', 0.40), ('annulus', 0.20), ('rectangle', 0.20), ('ellipse', 0.10), ('circle', 0.10)),
    model=TwoStageStackConfig(
        stage1=Stage1ModelConfig(hidden_layer_sizes=(256, 512, 256), dropout_rates=(0.15, 0.15, 0.15), training=StageTrainingConfig(epochs=220, batch_size=64, learning_rate=0.0004, validation_frequency=10, verbose=False, early_stopping_patience=12, lr_drop_factor=0.5, lr_drop_period=60, weight_decay=0.00001, gradient_clip_norm=1.0)),
        stage2=Stage2ModelConfig(hidden_layer_sizes=(512, 1024), dropout_rates=(0.10, 0.10), model_type='coord_conv_decoder', latent_grid_size=16, latent_channels=160, decoder_channels=(160, 128, 96, 64, 32), use_rectangle_edge_weighting=True, use_foreground_pos_weight=False, rectangle_edge_weight=4.0, rectangle_edge_width=3, edge_weight_mode='rectangle', annulus_edge_weight=1.0, annulus_edge_width=3, training=StageTrainingConfig(epochs=170, batch_size=96, learning_rate=0.0005, validation_frequency=60, verbose=False, early_stopping_patience=24, min_epochs=50, min_improvement=0.002, lr_drop_factor=0.5, lr_drop_period=80, weight_decay=0.00025, gradient_clip_norm=0.8, loss_type='bce_dice', dice_loss_weight=1.0, dice_smooth=1.0)),
    ),
    output_dir=OUTPUT_ROOT,
)

run_output_dir = run_config.run_output_dir
run_output_dir.mkdir(parents=True, exist_ok=True)
dataset_bundle = build_two_stage_datasets(run_config)
dataset_paths = save_two_stage_dataset(dataset_bundle, run_output_dir / 'datasets')

stage1_model = Stage1Regressor(run_config.gradient_feature_size, run_config.coefficient_size, run_config.model.stage1.hidden_layer_sizes, run_config.model.stage1.dropout_rates)
stage1_training = run_config.model.stage1.training
stage1_result = fit_stage1_model(model=stage1_model, train_features=dataset_bundle.train.gradient_data, train_targets=dataset_bundle.train.coefficients, val_features=dataset_bundle.validation.gradient_data, val_targets=dataset_bundle.validation.coefficients, epochs=stage1_training.epochs, batch_size=stage1_training.batch_size, learning_rate=stage1_training.learning_rate, device=DEVICE, validation_frequency=stage1_training.validation_frequency, verbose=stage1_training.verbose, early_stopping_patience=stage1_training.early_stopping_patience, lr_drop_factor=stage1_training.lr_drop_factor, lr_drop_period=stage1_training.lr_drop_period, weight_decay=stage1_training.weight_decay, gradient_clip_norm=stage1_training.gradient_clip_norm)

pred_train = predict_stage1_coefficients(stage1_model, dataset_bundle.train.gradient_data, DEVICE, stage1_result)
pred_val = predict_stage1_coefficients(stage1_model, dataset_bundle.validation.gradient_data, DEVICE, stage1_result)
pred_test = predict_stage1_coefficients(stage1_model, dataset_bundle.test.gradient_data, DEVICE, stage1_result)
pred_fixed = predict_stage1_coefficients(stage1_model, dataset_bundle.fixed.gradient_data, DEVICE, stage1_result)

def build_decoder():
    return Stage2CoordConvDecoder(run_config.coefficient_size, run_config.mask_pixels, run_config.model.stage2.hidden_layer_sizes, run_config.model.stage2.dropout_rates, run_config.model.stage2.latent_grid_size, run_config.model.stage2.latent_channels, run_config.model.stage2.decoder_channels)

def train_decoder(model, train_features, train_masks, val_features, val_masks, train_shape_types, use_rectangle_weighting):
    training = run_config.model.stage2.training
    return fit_stage2_model(model=model, train_features=train_features, train_targets=train_masks, val_features=val_features, val_targets=val_masks, epochs=training.epochs, batch_size=training.batch_size, learning_rate=training.learning_rate, device=DEVICE, validation_frequency=training.validation_frequency, verbose=training.verbose, early_stopping_patience=training.early_stopping_patience, train_shape_types=train_shape_types, grid_size=run_config.grid_size, use_rectangle_edge_weighting=use_rectangle_weighting, rectangle_edge_weight=run_config.model.stage2.rectangle_edge_weight, rectangle_edge_width=run_config.model.stage2.rectangle_edge_width, edge_weight_mode=run_config.model.stage2.edge_weight_mode, annulus_edge_weight=run_config.model.stage2.annulus_edge_weight, annulus_edge_width=run_config.model.stage2.annulus_edge_width, min_epochs=training.min_epochs, min_improvement=training.min_improvement, lr_drop_factor=training.lr_drop_factor, lr_drop_period=training.lr_drop_period, weight_decay=training.weight_decay, gradient_clip_norm=training.gradient_clip_norm, loss_type=training.loss_type, dice_loss_weight=training.dice_loss_weight, dice_smooth=training.dice_smooth, use_foreground_pos_weight=run_config.model.stage2.use_foreground_pos_weight)

def filter_shape(features, masks, shape_types, target):
    indices = [i for i, shape_type in enumerate(shape_types) if shape_type == target]
    return features[indices], masks[indices], tuple(shape_types[i] for i in indices)

general_model = build_decoder()
general_result = train_decoder(general_model, pred_train, dataset_bundle.train.masks, pred_val, dataset_bundle.validation.masks, dataset_bundle.train.shape_types, True)

ann_train_features, ann_train_masks, ann_train_types = filter_shape(pred_train, dataset_bundle.train.masks, dataset_bundle.train.shape_types, 'annulus')
ann_val_features, ann_val_masks, ann_val_types = filter_shape(pred_val, dataset_bundle.validation.masks, dataset_bundle.validation.shape_types, 'annulus')
annulus_model = build_decoder()
annulus_result = train_decoder(annulus_model, ann_train_features, ann_train_masks, ann_val_features, ann_val_masks, ann_train_types, False)

two_train_features, two_train_masks, two_train_types = filter_shape(pred_train, dataset_bundle.train.masks, dataset_bundle.train.shape_types, 'two_circles')
two_val_features, two_val_masks, two_val_types = filter_shape(pred_val, dataset_bundle.validation.masks, dataset_bundle.validation.shape_types, 'two_circles')
two_circles_model = build_decoder()
two_circles_result = train_decoder(two_circles_model, two_train_features, two_train_masks, two_val_features, two_val_masks, two_train_types, False)

class Router(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 3))
    def forward(self, inputs):
        return self.network(inputs)

def make_router_labels(shape_types):
    labels = []
    for shape_type in shape_types:
        if shape_type == 'annulus':
            labels.append(1)
        elif shape_type == 'two_circles':
            labels.append(2)
        else:
            labels.append(0)
    return np.array(labels, dtype=np.int64)

def train_router(train_features, train_shape_types, val_features, val_shape_types):
    router = Router(train_features.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(router.parameters(), lr=0.001, weight_decay=0.0001)
    criterion = nn.CrossEntropyLoss()
    train_x = torch.tensor(train_features, dtype=torch.float32, device=DEVICE)
    train_y = torch.tensor(make_router_labels(train_shape_types), dtype=torch.long, device=DEVICE)
    val_x = torch.tensor(val_features, dtype=torch.float32, device=DEVICE)
    val_y = torch.tensor(make_router_labels(val_shape_types), dtype=torch.long, device=DEVICE)
    best_state = None
    best_val_accuracy = -1.0
    history = {'train_loss': [], 'val_accuracy': []}
    for _ in range(100):
        router.train()
        optimizer.zero_grad()
        logits = router(train_x)
        loss = criterion(logits, train_y)
        loss.backward()
        optimizer.step()
        router.eval()
        with torch.no_grad():
            val_predictions = router(val_x).argmax(dim=1)
            val_accuracy = float((val_predictions == val_y).float().mean().item())
        history['train_loss'].append(float(loss.item()))
        history['val_accuracy'].append(val_accuracy)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in router.state_dict().items()}
    if best_state is not None:
        router.load_state_dict(best_state)
    return router, history, best_val_accuracy

router, router_history, router_val_accuracy = train_router(pred_train, dataset_bundle.train.shape_types, pred_val, dataset_bundle.validation.shape_types)

def routed_logits(features):
    general_logits = predict_stage2_logits(general_model, features, DEVICE, general_result)
    annulus_logits = predict_stage2_logits(annulus_model, features, DEVICE, annulus_result)
    two_circle_logits = predict_stage2_logits(two_circles_model, features, DEVICE, two_circles_result)
    with torch.no_grad():
        route_labels = router(torch.tensor(features, dtype=torch.float32, device=DEVICE)).argmax(dim=1).cpu().numpy()
    combined_logits = general_logits.copy()
    combined_logits[route_labels == 1] = annulus_logits[route_labels == 1]
    combined_logits[route_labels == 2] = two_circle_logits[route_labels == 2]
    return combined_logits, route_labels

val_logits, val_routes = routed_logits(pred_val)
test_logits, test_routes = routed_logits(pred_test)
fixed_logits, fixed_routes = routed_logits(pred_fixed)
threshold_summary = select_best_stage2_threshold(dataset_bundle.validation.masks, val_logits, run_config.threshold_candidates)
threshold_summary['selection_mode'] = 'validation_sweep'
threshold = float(threshold_summary['selected_threshold'])
summary = {
    'router_validation_accuracy': router_val_accuracy,
    'route_counts': {
        'validation': {'general': int((val_routes == 0).sum()), 'annulus': int((val_routes == 1).sum()), 'two_circles': int((val_routes == 2).sum())},
        'test': {'general': int((test_routes == 0).sum()), 'annulus': int((test_routes == 1).sum()), 'two_circles': int((test_routes == 2).sum())},
    },
    'stage1_metrics': {'test': evaluate_regression_predictions(dataset_bundle.test.coefficients, pred_test), 'fixed': evaluate_regression_predictions(dataset_bundle.fixed.coefficients, pred_fixed)},
    'general_metrics': {'test': evaluate_stage2_predictions(dataset_bundle.test.masks, predict_stage2_logits(general_model, pred_test, DEVICE, general_result), threshold), 'fixed': evaluate_stage2_predictions(dataset_bundle.fixed.masks, predict_stage2_logits(general_model, pred_fixed, DEVICE, general_result), threshold)},
    'routed_metrics': {'test': evaluate_stage2_predictions(dataset_bundle.test.masks, test_logits, threshold), 'fixed': evaluate_stage2_predictions(dataset_bundle.fixed.masks, fixed_logits, threshold)},
    'routed_metrics_by_shape': {'test': evaluate_stage2_predictions_by_shape(dataset_bundle.test.masks, test_logits, threshold, dataset_bundle.test.shape_types), 'fixed': evaluate_stage2_predictions_by_shape(dataset_bundle.fixed.masks, fixed_logits, threshold, dataset_bundle.fixed.shape_types)},
    'threshold_summary': threshold_summary,
    'dataset_paths': {key: str(value) for key, value in dataset_paths.items()},
    'stage1_history': stage1_result.history,
    'general_history': general_result.history,
    'annulus_history': annulus_result.history,
    'two_circles_history': two_circles_result.history,
    'router_history': router_history,
}
with (run_output_dir / 'summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)
print('General test IoU:', summary['general_metrics']['test']['mean_iou'])
print('Routed test IoU:', summary['routed_metrics']['test']['mean_iou'])
print('Router validation accuracy:', router_val_accuracy)
